# 13 — Cam-CAN → HCP: train on one cohort, test on the other

The model is fitted on **every** Cam-CAN subject and applied once to the HCP subjects it has never
seen. One number per configuration, scored against HCP's own questionnaire.

**Why not more cross-validation.** `select-bstm`'s five folds each estimate as many coefficients as
the whole model has, which is what makes 168 subjects a poor *training* cohort and a decent *test*
set. Simulated with this pipeline's own estimator, the same 168 subjects give about **11%** power to
detect a true R² of 0.20 as a training cohort, and about **83%** power to detect ρ = 0.20 as a test
set. Splitting training from testing uses each cohort for what it can do.

**No equating of the two questionnaires is needed.** Spearman uses ranks. The prediction comes out
in Cam-CAN's units and the truth is in the ASR's, and the two are never compared as values — only as
order. "Does the model rank these people the way their own instrument does" is a far weaker
assumption than making HADS and the ASR commensurate, which nothing here could support: the
instruments share no items and no person took both.

**What a null here would and would not mean.** Three things must hold for a score to appear — the
state space must carry over from 3T to 7T, the model from one age range to another, and the two
questionnaires must measure the same thing in the same direction. A zero does not say which failed.
So **the within-Cam-CAN result stays the primary finding and this is confirmation**; if Cam-CAN's own
CV is null, a null here adds nothing.

---

| section | question |
|---|---|
| **1** | are the two cohorts comparable at all? *(precondition — before any score)* |
| **2** | the whole multiverse, transferred |
| **3** | one model in depth |

## 0. Setup

Two `Settings`: one per cohort. They differ in the cohort, the task labels, the phenotype file and
the target columns — and `target_range`, because Cam-CAN codes "no answer" as **9999** in
`additional_HADS_depression` while HADS runs 0–21. That value is invisible while the target is the
`*_category` column (text never codes) and an enormous outlier the moment the raw column is used.

**Both targets are run**: the ordinal category the report has always used, and the raw score.
The raw has 22 levels against the category's 4, so it gives the model far more to rank — at the cost
of not being what the existing results used.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(REPO / "notebooks"), str(REPO)]

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import hads_tables as ht, hads_plots as hp
import cohort_plots as cp, cohort_tables as ct
import transfer_tables as tt

# ---- paths and names -------------------------------------------------------
CAMCAN   = Path("/project/6008063/tamires/cohorts/camcan/dataman/useraccess/"
                "opendata/paule_toussaint_camcan01870")
HCP_PHENO = Path("/project/6008063/tamires/cohorts/hcp_pheno/participants_pheno.csv")
SAVE_DIR = None          # e.g. REPO / "notebooks" / "figures" / "13"

# Which Cam-CAN target. Run the notebook twice, or set both and compare.
TARGET = "raw"           # "raw" | "category"

CC_TARGETS = {"raw":      {"anxiety": "additional_HADS_anxiety",
                           "depression": "additional_HADS_depression"},
              "category": {"anxiety": "additional_HADS_anx_category",
                           "depression": "additional_HADS_dep_category"}}[TARGET]

# HCP's own instrument. DSM_Anxi pairs with HADS anxiety and DSM_Depr with HADS
# depression; the LABELS must match between the two Settings, the columns need
# not -- Spearman only needs the two to order people the same way.
HCP_TARGETS = {"anxiety": "DSM_Anxi_Raw", "depression": "DSM_Depr_Raw"}

# HCP spells the covariates differently: Gender for Sex, Age_in_Yrs for Age.
# Lined up by position instead of by name, the model would read sex as
# something else and every number downstream would still look reasonable --
# which is why transfer_tables refuses rather than guesses.
HCP_ALIASES = {"Sex": "Gender", "Age": "Age_in_Yrs"}

In [ ]:
train_cfg = ht.Settings(
    cohorts=["camcan", "camcan_rest"],
    tasks={"rest": "rest", "movie": "movie"},
    targets=CC_TARGETS,
    camcan=CAMCAN,
    # only the raw columns have a scale to be outside of
    target_range=({"anxiety": (0, 21), "depression": (0, 21)}
                  if TARGET == "raw" else None),
    covariates=["Age", "Sex", "n_transitions"],
    baseline_covariates=["Age", "Sex"],
    save_dir=SAVE_DIR,
)

test_cfg = ht.Settings(
    cohorts=["hcp7t_fix"],
    # the four movies and the four rests are POOLED below, so these are the
    # labels the pooled rows carry
    tasks={"rest": "REST", "movie": "MOVIE"},
    targets=HCP_TARGETS,
    pheno_files=[(HCP_PHENO, ",")],
    pheno_id="Subject",
    covariates=["Age", "Sex", "n_transitions"],
    baseline_covariates=["Age", "Sex"],
    save_dir=SAVE_DIR,
)

train, test = ht.Study(train_cfg), ht.Study(test_cfg)
print(f"train  {len(train.pheno):,} subjects with phenotype")
print(f"test   {len(test.pheno):,} subjects with phenotype")

### Pooling HCP's four runs

HCP gives MOVIE1–4 and REST1–4 per subject; the models score **one row per subject** and will not
average them. Averaging the columns would be wrong for most: an entropy rate averaged over runs is
not the entropy rate of anything, a switch rate is a ratio whose denominators differ, and occupancy
is a fraction of a different number of windows each time.

`pool_tasks` recomputes instead. `p_joint × n_transitions` is the count matrix, and counts add, so
the cells, `n_transitions` and occupancy pool **exactly**; the rates and the entropy are then
recomputed from the pooled matrix. The one approximation is `dispersion`, which needs the latents
rather than this table — it is flagged in the frame.

In [ ]:
_load = ht.Study.load_set

def load_pooled(self, space, task):
    """load_set, with HCP's runs of one condition pooled into one row."""
    d = _load(self, space, task)
    if self.cfg is not test_cfg or d["sub"].is_unique:
        return d
    return tt.pool_tasks(d, ht.n_states(space["states"]), log=lambda *_: None)

ht.Study.load_set = load_pooled
print("load_set now pools the test cohort's runs")

## 1. Are the two cohorts comparable at all?

**A precondition, not a result.** If HCP's brain states do not behave like Cam-CAN's, a model
trained on one cannot transfer to the other, and the score below would be measuring that rather than
anything about symptoms. These are notebook 11's figures with HCP in place of the other cohorts.

In [ ]:
# cohort_tables has its OWN Settings, keyed by one state set and a cohort->dirs
# map, not hads_tables'. Pick the space section 2 ends up caring about; this
# one is a precondition check, so a representative set is enough.
co_cfg = ct.Settings(
    space=dict(atlas="yeo7", window_s=-1.0, states="HMM1_pca3_8"),
    cohorts={"ds002837":  dict(dirs=["ds002837"],  role="train"),
             "cneuromod": dict(dirs=["cneuromod"], role="train"),
             "camcan":    dict(dirs=["camcan", "camcan_rest"], role="projected"),
             "hcp7t_fix": dict(dirs=["hcp7t_fix"], role="projected")},
    save_dir=SAVE_DIR,
)
co = ct.Cohorts(co_cfg)
agree = co.agreement()
cp.fig_tost(co, agree)          # 2a: is every cohort equivalent to the others?

In [ ]:
groups = co.groups("cohort")
cp.fig_matrices(co, groups, "tost", title="4a  transitions by cohort",
                name="13_4a_matrix_cohort")
cp.fig_matrices(co, groups, "cohen", sd=True, title="4b  transitions by cohort",
                name="13_4b_matrix_cohort")

## 2. The whole multiverse, transferred

Every configuration the two cohorts share: fit on all of Cam-CAN, one score on HCP.

**The error bar means something different here.** In the Cam-CAN report `± sd` is the spread over
fold seeds — how much the answer moves when the folds are redrawn. There are no folds now. It is the
bootstrap sd over the **test subjects**: how much the answer moves when a different 168 people are
tested. Both are error bars; they answer different questions.

The baseline is Age + Sex trained on Cam-CAN and applied to HCP the same way, so the comparison is
like for like.

In [ ]:
raw_rows = tt.transfer_rows(train, test, n_boot=2000, seed=0)
rows = tt.as_report_rows(raw_rows)
print(f"{len(rows)} configuration(s);  "
      f"{rows['over_base'].mean():.0%} above their transferred baseline")
rows.sort_values("score", ascending=False).head(12)[
    ["atlas", "window", "states", "task", "phenotype", "arm", "model",
     "score", "lo", "hi", "base_score"]]

### The fcm half

Static FC transferred the same way, so the four tables can be compared. Two decisions have to come
from the **training** cohort alone, and both would be invisible if they went wrong:

- **which edges.** `fcm_selection` drops an edge that is NaN in more than `max_missing` of subjects,
  so the surviving set follows each cohort's own coverage. Let HCP choose its own and the model is
  handed a different feature space than it was fitted on — with the columns still lining up by count.
- **the imputed value.** A NaN is filled with a column mean. Filled from HCP's mean, every test
  subject's missing edge is nudged toward HCP's centre: adaptation to the data being scored.

Pooling HCP's four runs is **not exact here**, unlike on the transition side. A correlation of four
concatenated runs is not a function of the four correlations — each was computed against its own
run's mean. What is done is the standard meta-analytic pooling: a mean of Fisher-z values weighted
by `n_tr_used`. The exact alternative would be `static-fc --pool` taking a task filter, which does
not exist.

In [ ]:
fcm_raw = tt.fcm_transfer_rows(
    train, test,
    train_tasks={"rest": "rest", "movie": "movie"},
    test_tasks={"rest": "REST", "movie": "MOVIE"},   # prefix: pools REST1-4 / MOVIE1-4
    covariates=["Age", "Sex", "frac_good_frames", "n_tr_used"],
    n_boot=2000, seed=0)
fcm = tt.as_report_rows(fcm_raw)
print(f"{len(fcm)} fcm configuration(s);  "
      f"{fcm['over_base'].mean():.0%} above their transferred baseline"
      if len(fcm) else "no static-FC table for both cohorts yet")
display(fcm[["atlas", "task", "phenotype", "arm", "model", "score", "lo", "hi",
             "base_score", "n_edges_used", "n_test"]])

# the four tables together: bstm and fcm, rest and movie
rows = pd.concat([rows, fcm], ignore_index=True)

In [ ]:
hp.fig_rankings(rows, train_cfg, train.above_both)      # B1, transferred

In [ ]:
hp.fig_factors(rows, train_cfg)                         # B2, transferred

In [ ]:
hp.fig_k_curve(rows, train_cfg)                         # B2.K, transferred

## 3. One model in depth

Pick the configuration to look at. The default is the best transferred score, but a row that only
just cleared its baseline on a lucky bootstrap is not a finding — prefer one whose **interval**
clears it.

In [ ]:
clears = rows[rows["lo"] > rows["base_score"]].sort_values("score", ascending=False)
print(f"{len(clears)} of {len(rows)} configuration(s) have an interval clearing their baseline")
best = (clears if len(clears) else rows.sort_values("score", ascending=False)).iloc[0]
space = {"atlas": best["atlas"], "window_s": best["window_s"],
         "states": best["states"]}
print(f"chosen: {best['label']}  ·  {best['arm']}  ·  {best['model']}")
print(f"  score {best['score']:+.3f}  [{best['lo']:+.3f}, {best['hi']:+.3f}]"
      f"   baseline {best['base_score']:+.3f}")
if not len(clears):
    print("  !! nothing clears its baseline -- this row is the best of a null set")

### E1 — how the chosen set scores, bstm beside fcm

Both trees are in `rows` now, so this is notebook 10's E1 comparison: rest against movie, anxiety
against depression, brain-state transitions against static FC — each over its own transferred
baseline.

The fcm rows have no aperture and no state set: static FC is one matrix per subject over every
usable frame, the still twin of `window_s = -1`.

In [ ]:
sel = ((rows["states"] == best["states"]) & (rows["atlas"] == best["atlas"])
       & (rows["arm"] == best["arm"]) & (rows["model"] == best["model"]))
# the fcm twin of the same atlas and model, whichever arm scored best there
fcm_sel = rows["tree"].eq("fcm") & rows["atlas"].eq(best["atlas"]) & \
          rows["model"].eq(best["model"])
perf = rows[sel | fcm_sel]
display(perf[["tree", "arm", "task", "phenotype", "score", "lo", "hi",
              "base_score", "n_test"]])

fig, axes = plt.subplots(1, len(train_cfg.targets), squeeze=False,
                         figsize=(4.6 * len(train_cfg.targets), 3.8), sharey=True)
for ax, lab in zip(axes[0], train_cfg.targets):
    d = perf[perf["phenotype"] == lab].sort_values(["tree", "task"])
    for k, (_, r) in enumerate(d.iterrows()):
        c = hp.TASK_COLOURS[list(train_cfg.tasks).index(r["task"])
                            % len(hp.TASK_COLOURS)]
        ax.plot([k, k], [r["lo"], r["hi"]], color=c, lw=3, solid_capstyle="round")
        ax.plot(k, r["score"], "o" if r["tree"] == "bstm" else "D", ms=9,
                color=c, mec=hp.SURFACE, mew=1.5, zorder=3)
        ax.plot([k - .3, k + .3], [r["base_score"]] * 2, color=hp.INK2, ls="--", lw=1)
    ax.axhline(0, color=hp.MUTED, lw=.6)
    ax.set_xticks(range(len(d)),
                  [f"{t}\n{k}" for t, k in zip(d["tree"], d["task"])], fontsize=8)
    ax.set_title(lab, loc="left")
axes[0, 0].set_ylabel("transfer score on HCP (Spearman)")
fig.suptitle(f"E1  transferred · {best['label']} · {best['model']}",
             x=.01, ha="left", fontsize=10, color=hp.INK)
fig.text(.01, -.05,
         "circle = bstm, diamond = fcm; bar = 95% bootstrap over the test subjects; "
         "dashed = Age + Sex trained on Cam-CAN and applied the same way",
         fontsize=7.5, color=hp.INK2)
fig.tight_layout()
plt.show()

### E3 — where the model looks

SHAP of the model **trained on Cam-CAN**, which is notebook 10's E3 unchanged: nothing about that
model differs here, only where it is applied. Rather than duplicate notebook 10's setup — `deep`,
`model_marks`, the reference and colour maps — open **notebook 10 section E3** with
`SPACE` set to the space printed above.

What this notebook adds is the reason to look: read E3 beside section 1. A model leaning on a
transition whose two cohorts are **not** equivalent is a model whose transfer was never going to
work, and that is visible by putting the two figures side by side.

In [ ]:
print("open notebook 10 at E3 with:")
print(f'  SPACE = dict(atlas="{best["atlas"]}", '
      f'window_s={best["window_s"]}, states="{best["states"]}")')
print(f'  E1_MODEL = "{best["model"]}"     # arm: {best["arm"]}')

## What to write down

1. **Section 1 first.** If the cohorts are not equivalent, sections 2 and 3 are measuring the
   mismatch.
2. **An interval that clears the baseline** is the claim. A point estimate above it with an interval
   that crosses is 168 subjects, not a finding.
3. **A null is ambiguous** — state space, model, or instrument, and this cannot separate them. Say
   so rather than concluding the transitions do not predict.
4. Run with `TARGET = "raw"` and `TARGET = "category"` and compare. The raw has 22 levels against 4,
   so it should rank better if the signal is real and worse if the category was doing the work.